In [8]:
import onnx

model_path = r'C:/Users/swest/Desktop/traffic-capturing/traffic-reporting/models/weights/silent_contract_9082_original_640px_bs416.onnx'

try:
    # Load the ONNX model
    model = onnx.load(model_path)

    input_names = [inp.name for inp in model.graph.input]

    print(f"Definitive Model Input Name(s) found in '{model_path}':")
    if not input_names:
        print("No primary inputs found. This might indicate an issue with the model or it might only have initializer inputs.")
    else:
        for name in input_names:
            print(f"- {name}")

    # For more details, you can print shape information if available
    print("\nDetailed Input Information:")
    for i, input_tensor in enumerate(model.graph.input):
        print(f"\nInput #{i}:")
        print(f"  Name: {input_tensor.name}")
        # The shape is under type.tensor_type.shape.dim
        dims = input_tensor.type.tensor_type.shape.dim
        shape_str = ", ".join([str(dim.dim_value) if dim.dim_value > 0 else dim.dim_param if dim.dim_param else "?" for dim in dims])
        print(f"  Shape (from model): [{shape_str}]")
        print(f"  Data Type (from model): {onnx.TensorProto.DataType.Name(input_tensor.type.tensor_type.elem_type)}")

except Exception as e:
    print(f"An error occurred: {e}")
    print("Please ensure the model path is correct and the ONNX file is not corrupted.")

Definitive Model Input Name(s) found in 'C:/Users/swest/Desktop/traffic-capturing/traffic-reporting/models/weights/silent_contract_9082_original_640px_bs416.onnx':
- images

Detailed Input Information:

Input #0:
  Name: images
  Shape (from model): [batch, 3, height, width]
  Data Type (from model): FLOAT16


In [1]:
import onnx

model = onnx.load('C:/Users/swest/Desktop/traffic-capturing/traffic-reporting/models/weights/silent_contract_9082_original_640px_bs416.onnx')

# Check input dimensions
for input in model.graph.input:
    print(f"Input: {input.name}")
    for dim in input.type.tensor_type.shape.dim:
        if dim.dim_param:
            print(f"  Dynamic dimension: {dim.dim_param}")
        else:
            print(f"  Fixed dimension: {dim.dim_value}")


Input: images
  Dynamic dimension: batch
  Fixed dimension: 3
  Dynamic dimension: height
  Dynamic dimension: width


In [7]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession('C:/Users/swest/Desktop/traffic-capturing/traffic-reporting/models/weights/silent_contract_9082_original_640px_bs416.onnx', providers=['CUDAExecutionProvider'])

# Test batch size 1 with correct dtype
input_1 = np.random.randn(1, 3, 640, 640).astype(np.float16)  # Changed to float16
output_1 = session.run(None, {'images': input_1})

# Test batch size 4
input_4 = np.random.randn(4, 3, 640, 640).astype(np.float16)  # Changed to float16
output_4 = session.run(None, {'images': input_4})
print("Dynamic batch size confirmed!")
print(f"Batch 1 output shape: {output_1[0].shape}")
print(f"Batch 4 output shape: {output_4[0].shape}")


Dynamic batch size confirmed!
Batch 1 output shape: (1, 84, 8400)
Batch 4 output shape: (4, 84, 8400)
